# My version of the finished M1 functions

In [1]:
!pip install networkx
!pip install transformers
!pip install PyPDF2
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.9 MB/s eta 0:00:00


In [2]:
import os
import torch
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tqdm import tqdm

from transformers import pipeline
from PyPDF2 import PdfReader
from collections import Counter
import re
from wordcloud import WordCloud
from google.colab import files

In [4]:
def upload_pdf():
    """
    Upload a PDF file using Google Colab's file upload functionality.

    Returns:
    str: Path to the uploaded PDF file
    """
    print("Please upload a PDF file:")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No file was uploaded. Please try again.")

    # Get the filename of the uploaded file
    pdf_path = list(uploaded.keys())[0]
    print(f"Uploaded file: {pdf_path}")
    return pdf_path

In [5]:
# PDF Text Extraction
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    text = ""
    for page in tqdm(reader.pages,
                     desc="📄 Extracting PDF Text",
                     total=total_pages,
                     unit="page",
                     colour="green"):
        text += page.extract_text() + "\n"
    return text

In [6]:
# Keyword Extraction
def extract_advanced_keywords(text, top_k=20):
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    stop_words = set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'])
    filtered_words = [word for word in words if word not in stop_words and len(word) > 2]
    word_freq = Counter(filtered_words)

    def calculate_importance(word):
        freq_score = word_freq[word]
        length_score = len(word)
        rarity_score = 1 / (word_freq[word] + 1)
        return freq_score * length_score * rarity_score

    scored_keywords = [
        {"keyword": word, "score": calculate_importance(word), "frequency": word_freq[word]}
        for word in set(filtered_words)
    ]
    scored_keywords.sort(key=lambda x: x['score'], reverse=True)
    return scored_keywords[:top_k]

In [7]:
def create_knowledge_graph(keywords):
    G = nx.Graph()
    max_score = max(kw['score'] for kw in keywords)
    max_freq = max(kw['frequency'] for kw in keywords)

    for keyword in tqdm(keywords, desc="🌐 Creating Knowledge Graph Nodes", unit="node", colour="yellow"):
        # Color intensity signifies importance
        color_intensity = 0.2 + (keyword['score'] / max_score) * 0.8
        node_color = plt.cm.Blues(color_intensity)

        # Node size signifies frequency
        node_size = 100 + (keyword['frequency'] / max_freq) * 25000  # Signifies the frequency of the keywords

        G.add_node(keyword['keyword'], size=node_size, color=node_color, frequency=keyword['frequency'])

    keywords_list = [k['keyword'] for k in keywords]
    for i in tqdm(range(len(keywords_list)), desc="🔗 Creating Graph Edges", unit="edge", colour="magenta"):
        for j in range(i+1, len(keywords_list)):
            G.add_edge(keywords_list[i], keywords_list[j])

    return G

In [8]:
# Knowledge Graph Visualization
def visualize_knowledge_graph(graph, output_path='knowledge_graph.png'):
    plt.figure(figsize=(20, 14))
    pos = nx.spring_layout(graph, k=0.5, iterations=50)
    node_sizes = [graph.nodes[node]['size'] for node in graph.nodes()]
    node_colors = [graph.nodes[node]['color'] for node in graph.nodes()]
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8)
    nx.draw_networkx_edges(graph, pos, width=1, alpha=0.5, edge_color='gray')
    nx.draw_networkx_labels(graph, pos, font_size=8, font_weight="bold")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📊 Knowledge graph visualization saved to {output_path}")

In [9]:
# EDA Visualizations
def plot_keyword_frequencies(keywords, output_path='keyword_frequencies.png'):
    keywords_sorted = sorted(keywords, key=lambda x: x['frequency'], reverse=True)
    keywords_list = [kw['keyword'] for kw in keywords_sorted]
    frequencies = [kw['frequency'] for kw in keywords_sorted]
    plt.figure(figsize=(12, 6))
    plt.bar(keywords_list, frequencies, color='skyblue')
    plt.xlabel('Keywords', fontsize=14)
    plt.ylabel('Frequency', fontsize=14)
    plt.title('Keyword Frequency Distribution', fontsize=16, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"📊 Keyword frequency bar chart saved to {output_path}")

def plot_keyword_importance(keywords, output_path='keyword_importance.png'):
    keywords_list = [kw['keyword'] for kw in keywords]
    frequencies = [kw['frequency'] for kw in keywords]
    importance_scores = [kw['score'] for kw in keywords]
    plt.figure(figsize=(12, 6))
    plt.scatter(frequencies, importance_scores, s=100, color='blue', alpha=0.7)
    for i, keyword in enumerate(keywords_list):
        plt.text(frequencies[i], importance_scores[i], keyword, fontsize=9, ha='right', va='bottom')
    plt.xlabel('Frequency', fontsize=14)
    plt.ylabel('Importance Score', fontsize=14)
    plt.title('Keyword Importance vs. Frequency', fontsize=16, fontweight='bold')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"📊 Keyword importance scatter plot saved to {output_path}")

def generate_wordcloud(keywords, output_path='keyword_wordcloud.png'):
    word_freq = {kw['keyword']: kw['frequency'] for kw in keywords}
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(word_freq)
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Keyword Word Cloud', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"☁️ Word cloud saved to {output_path}")

In [10]:
# Main Pipeline
def main():
    with tqdm(total=6, desc="🚀 PDF Keyword Analysis Pipeline", colour="green") as main_pbar:
        pdf_path = upload_pdf()  # Use the new upload function
        main_pbar.update(1)
        pdf_text = extract_text_from_pdf(pdf_path)
        main_pbar.update(1)
        keywords = extract_advanced_keywords(pdf_text, top_k=20)
        main_pbar.update(1)
        knowledge_graph = create_knowledge_graph(keywords)
        main_pbar.update(1)
        print("\n🏷️ Extracted Keywords:")
        for kw in keywords:
            print(f"{kw['keyword']} (Importance Score: {kw['score']:.2f}, Frequency: {kw['frequency']})")
        visualize_knowledge_graph(knowledge_graph)
        main_pbar.update(1)
        plot_keyword_frequencies(keywords)
        plot_keyword_importance(keywords)
        generate_wordcloud(keywords)
        main_pbar.update(2)

if __name__ == "__main__":
    main()

🚀 PDF Keyword Analysis Pipeline:   0%|          | 0/6 [00:00<?, ?it/s]

Please upload a PDF file:


🚀 PDF Keyword Analysis Pipeline:  17%|█▋        | 1/6 [00:52<04:20, 52.10s/it]

Saving Economics Manag Strategy - March 1996 - Brandenburger - Value‐based Business Strategy.pdf to Economics Manag Strategy - March 1996 - Brandenburger - Value‐based Business Strategy.pdf
Uploaded file: Economics Manag Strategy - March 1996 - Brandenburger - Value‐based Business Strategy.pdf



🌐 Creating Knowledge Graph Nodes: 100%|██████████| 20/20 [00:00<00:00, 2522.66node/s]

🔗 Creating Graph Edges: 100%|██████████| 20/20 [00:00<00:00, 52200.42edge/s]



🏷️ Extracted Keywords:
noncooperative (Importance Score: 12.73, Frequency: 10)
onlinelibrary (Importance Score: 12.67, Frequency: 38)
appropriation (Importance Score: 11.92, Frequency: 11)
opportunities (Importance Score: 11.70, Frequency: 9)
unrestricted (Importance Score: 11.14, Frequency: 13)
willingness (Importance Score: 10.73, Frequency: 39)
opportunity (Importance Score: 10.69, Frequency: 34)
characterization (Importance Score: 10.67, Frequency: 2)
cooperative (Importance Score: 10.35, Frequency: 16)
competitors (Importance Score: 10.08, Frequency: 11)
competition (Importance Score: 9.90, Frequency: 9)
appropriate (Importance Score: 9.78, Frequency: 8)
conditions (Importance Score: 9.76, Frequency: 41)
underpinnings (Importance Score: 9.75, Frequency: 3)
brandenburger (Importance Score: 9.75, Frequency: 3)
bargaining (Importance Score: 9.74, Frequency: 38)
specifically (Importance Score: 9.60, Frequency: 4)
organization (Importance Score: 9.60, Frequency: 4)
university (Importa

🚀 PDF Keyword Analysis Pipeline:  83%|████████▎ | 5/6 [00:55<00:06,  6.79s/it]

📊 Knowledge graph visualization saved to knowledge_graph.png
📊 Keyword frequency bar chart saved to keyword_frequencies.png
📊 Keyword importance scatter plot saved to keyword_importance.png


🚀 PDF Keyword Analysis Pipeline: 7it [00:58,  8.37s/it]

☁️ Word cloud saved to keyword_wordcloud.png


# Creating Graphs without file upload function (something that can be integrated with our Step 1):

In [ ]:
!pip install networkx
!pip install transformers
!pip install PyPDF2
!pip install sentence_transformers

In [ ]:
import os
import torch
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tqdm import tqdm

from transformers import pipeline
from PyPDF2 import PdfReader
from collections import Counter
import re
from wordcloud import WordCloud

In [ ]:
# PDF Text Extraction
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)
    text = ""
    for page in tqdm(reader.pages,
                     desc="📄 Extracting PDF Text",
                     total=total_pages,
                     unit="page",
                     colour="green"):
        text += page.extract_text() + "\n"
    return text

# Keyword Extraction
def extract_advanced_keywords(text, top_k=20):
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    stop_words = set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'])
    filtered_words = [word for word in words if word not in stop_words and len(word) > 2]
    word_freq = Counter(filtered_words)

    def calculate_importance(word):
        freq_score = word_freq[word]
        length_score = len(word)
        rarity_score = 1 / (word_freq[word] + 1)
        return freq_score * length_score * rarity_score

    scored_keywords = [
        {"keyword": word, "score": calculate_importance(word), "frequency": word_freq[word]}
        for word in set(filtered_words)
    ]
    scored_keywords.sort(key=lambda x: x['score'], reverse=True)
    return scored_keywords[:top_k]

def create_knowledge_graph(keywords):
    G = nx.Graph()
    max_score = max(kw['score'] for kw in keywords)
    max_freq = max(kw['frequency'] for kw in keywords)

    for keyword in tqdm(keywords, desc="🌐 Creating Knowledge Graph Nodes", unit="node", colour="yellow"):
        # Color intensity signifies importance
        color_intensity = 0.2 + (keyword['score'] / max_score) * 0.8
        node_color = plt.cm.Blues(color_intensity)

        # Node size signifies frequency
        node_size = 100 + (keyword['frequency'] / max_freq) * 25000  # Signifies the frequency of the keywords

        G.add_node(keyword['keyword'], size=node_size, color=node_color, frequency=keyword['frequency'])

    keywords_list = [k['keyword'] for k in keywords]
    for i in tqdm(range(len(keywords_list)), desc="🔗 Creating Graph Edges", unit="edge", colour="magenta"):
        for j in range(i+1, len(keywords_list)):
            G.add_edge(keywords_list[i], keywords_list[j])

    return G

# Knowledge Graph Visualization
def visualize_knowledge_graph(graph, output_path='knowledge_graph.png'):
    plt.figure(figsize=(20, 14))
    pos = nx.spring_layout(graph, k=0.5, iterations=50)
    node_sizes = [graph.nodes[node]['size'] for node in graph.nodes()]
    node_colors = [graph.nodes[node]['color'] for node in graph.nodes()]
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8)
    nx.draw_networkx_edges(graph, pos, width=1, alpha=0.5, edge_color='gray')
    nx.draw_networkx_labels(graph, pos, font_size=8, font_weight="bold")
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📊 Knowledge graph visualization saved to {output_path}")

# EDA Visualizations
def plot_keyword_frequencies(keywords, output_path='keyword_frequencies.png'):
    keywords_sorted = sorted(keywords, key=lambda x: x['frequency'], reverse=True)
    keywords_list = [kw['keyword'] for kw in keywords_sorted]
    frequencies = [kw['frequency'] for kw in keywords_sorted]
    plt.figure(figsize=(12, 6))
    plt.bar(keywords_list, frequencies, color='skyblue')
    plt.xlabel('Keywords', fontsize=14)
    plt.ylabel('Frequency', fontsize=14)
    plt.title('Keyword Frequency Distribution', fontsize=16, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"📊 Keyword frequency bar chart saved to {output_path}")

def plot_keyword_importance(keywords, output_path='keyword_importance.png'):
    keywords_list = [kw['keyword'] for kw in keywords]
    frequencies = [kw['frequency'] for kw in keywords]
    importance_scores = [kw['score'] for kw in keywords]
    plt.figure(figsize=(12, 6))
    plt.scatter(frequencies, importance_scores, s=100, color='blue', alpha=0.7)
    for i, keyword in enumerate(keywords_list):
        plt.text(frequencies[i], importance_scores[i], keyword, fontsize=9, ha='right', va='bottom')
    plt.xlabel('Frequency', fontsize=14)
    plt.ylabel('Importance Score', fontsize=14)
    plt.title('Keyword Importance vs. Frequency', fontsize=16, fontweight='bold')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"📊 Keyword importance scatter plot saved to {output_path}")

def generate_wordcloud(keywords, output_path='keyword_wordcloud.png'):
    word_freq = {kw['keyword']: kw['frequency'] for kw in keywords}
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(word_freq)
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Keyword Word Cloud', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"☁️ Word cloud saved to {output_path}")

# Main Pipeline
def main(pdf_path):
    with tqdm(total=6, desc="🚀 PDF Keyword Analysis Pipeline", colour="green") as main_pbar:
        pdf_text = extract_text_from_pdf(pdf_path)
        main_pbar.update(1)
        keywords = extract_advanced_keywords(pdf_text, top_k=20)
        main_pbar.update(1)
        knowledge_graph = create_knowledge_graph(keywords)
        main_pbar.update(1)
        print("\n🏷️ Extracted Keywords:")
        for kw in keywords:
            print(f"{kw['keyword']} (Importance Score: {kw['score']:.2f}, Frequency: {kw['frequency']})")
        visualize_knowledge_graph(knowledge_graph)
        main_pbar.update(1)
        plot_keyword_frequencies(keywords)
        plot_keyword_importance(keywords)
        generate_wordcloud(keywords)
        main_pbar.update(2)

if __name__ == "__main__":
    pdf_path = "xxxxxxxxxxx.pdf" # Change to match the desired file
    main(pdf_path)